# 🎯 Estandarización: Clase `pokemon`
**ExpoEscom — Clasificador Multietiqueta**

**Qué hace este notebook:**
- Accede a la carpeta compartida de Pokemon en Drive
- Extrae **solo N imágenes** de los archivos ZIP/RAR (no descomprime todo)
- Las estandariza a 224x224 JPG
- Las copia a tu dataset unificado en `MyDrive/ExpoEscom/dataset/pokemon/`

**⚠️ Importante:** Los RARs en Pokemon pesan ~22GB en total. 
Este notebook es inteligente: **para en cuanto tiene las N imágenes necesarias.**

## CELDA 1 — Instalar dependencias y montar Drive

In [ ]:
# ============================================================
# Dependencias: unrar (para .rar), Pillow ya viene en Colab
# ============================================================
!apt-get install -y unrar > /dev/null 2>&1
print('✅ unrar instalado')

import os, shutil, zipfile, subprocess, random, time
from pathlib import Path
from PIL import Image
import io
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print('✅ Drive montado')

## CELDA 2 — Configuración (solo modifica aquí)

In [ ]:
# ════ CONFIGURACIÓN ═══════════════════════════════════════════

# Carpeta FUENTE: la carpeta compartida del equipo Pokemon
# (ya la tienes en 'Compartidos conmigo' → se accede por ID)
FOLDER_ID_POKEMON = '1LP8h59yPavCtmFgUtXr9hrdmg5ltQlaL'

# Ruta en tu Drive donde está el dataset unificado
DATASET_DESTINO = '/content/drive/MyDrive/ExpoEscom/dataset'

# Clase que estamos procesando
CLASE = 'pokemon'

# ── Para el MINIMODELO DE PRUEBA ─────────────────────────────
# Cambia a 100_000 cuando quieras el dataset completo
MAX_IMAGENES = 500   # imágenes para el mini PoC

# Directorio temporal en Colab (RAM rápida, se borra al cerrar sesión)
TMP_DIR = f'/content/tmp_{CLASE}'
TMP_EXTRACT = f'{TMP_DIR}/extract'

# Destino final
DESTINO_CLASE = os.path.join(DATASET_DESTINO, CLASE)

# ── Crear carpetas necesarias ────────────────────────────────
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(TMP_EXTRACT, exist_ok=True)
os.makedirs(DESTINO_CLASE, exist_ok=True)

print(f'Clase          : {CLASE}')
print(f'Imágenes meta  : {MAX_IMAGENES}')
print(f'Destino final  : {DESTINO_CLASE}')
print(f'Temp dir       : {TMP_DIR}')

## CELDA 3 — Diagnóstico: listar archivos en la carpeta compartida

In [ ]:
# ── Lista los archivos en la carpeta compartida via gdown ────
# Necesitamos gdown para acceder a carpetas compartidas
!pip install gdown -q
import gdown

print('═' * 55)
print(f'📁 Carpeta Pokemon: {FOLDER_ID_POKEMON}')
print('═' * 55)

# Listar contenido de la carpeta compartida
try:
    archivos = gdown.download_folder(
        id=FOLDER_ID_POKEMON,
        output=TMP_DIR,
        quiet=False,
        use_cookies=False,
        remaining_ok=True,
        skip_download=True   # Solo listar, NO descargar aún
    )
    print(f'\n✅ Archivos encontrados: {len(archivos) if archivos else 0}')
    if archivos:
        for a in archivos:
            print(f'   {a}')
except Exception as e:
    print(f'⚠️ Error listando con gdown: {e}')
    print('→ Prueba la CELDA 3B más abajo (método alternativo con PyDrive)')

In [ ]:
# ── CELDA 3B — Método alternativo: PyDrive2 ─────────────────
# Usa esto si la CELDA 3 falló

!pip install PyDrive2 -q
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Autenticación con tu cuenta Google
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive_api = GoogleDrive(gauth)

# Listar archivos en la carpeta compartida
query = f"'{FOLDER_ID_POKEMON}' in parents and trashed=false"
file_list = drive_api.ListFile({'q': query}).GetList()

print(f'Archivos en la carpeta Pokemon ({len(file_list)} total):')
print('─' * 60)
archivos_info = []
for f in file_list:
    size_mb = int(f.get('fileSize', 0)) / (1024*1024) if f.get('fileSize') else 0
    ext = Path(f['title']).suffix.lower()
    print(f"  {'📦' if ext in ['.zip','.rar'] else '📁'} {f['title']:45s} {size_mb:8.1f} MB  id:{f['id']}")
    archivos_info.append({
        'id': f['id'],
        'nombre': f['title'],
        'ext': ext,
        'size_mb': size_mb
    })

print(f'\nTotal: {sum(a["size_mb"] for a in archivos_info):.1f} MB')

## CELDA 4 — Funciones de estandarización

In [ ]:
# ── Funciones core ───────────────────────────────────────────

EXTENSIONES_IMAGEN = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif'}

def es_imagen(ruta):
    """Verifica si un archivo es una imagen por extensión."""
    return Path(ruta).suffix.lower() in EXTENSIONES_IMAGEN


def estandarizar_imagen(ruta_origen, ruta_destino, size=(224, 224)):
    """
    Convierte cualquier imagen a JPG 224x224 RGB.
    Retorna True si tuvo éxito, False si falló.
    """
    try:
        with Image.open(ruta_origen) as img:
            # Convertir a RGB (maneja RGBA, paleta, escala de grises, etc.)
            img = img.convert('RGB')
            # Resize con LANCZOS (mejor calidad)
            img = img.resize(size, Image.LANCZOS)
            # Guardar como JPG con calidad 90
            img.save(ruta_destino, 'JPEG', quality=90, optimize=True)
        return True
    except Exception as e:
        return False


def extraer_de_zip(zip_path, destino_extract, max_imgs, contador_global):
    """
    Extrae imágenes de un ZIP de forma inteligente:
    - Para en cuanto alcanza max_imgs totales
    - Retorna el número de imágenes procesadas
    """
    procesadas = 0
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            # Filtrar solo archivos de imagen dentro del ZIP
            imgs_en_zip = [
                name for name in zf.namelist()
                if es_imagen(name) and not name.startswith('__MACOSX')
            ]
            random.shuffle(imgs_en_zip)  # Mezclar para no tomar solo del inicio
            
            for nombre in imgs_en_zip:
                if contador_global[0] >= max_imgs:
                    break
                
                try:
                    # Leer imagen directamente desde el ZIP (sin extraer a disco)
                    with zf.open(nombre) as img_file:
                        img_bytes = img_file.read()
                    
                    # Generar nombre único para evitar colisiones
                    idx = contador_global[0]
                    dest_path = os.path.join(DESTINO_CLASE, f'{CLASE}_{idx:06d}.jpg')
                    
                    # Estandarizar y guardar
                    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
                    img = img.resize((224, 224), Image.LANCZOS)
                    img.save(dest_path, 'JPEG', quality=90, optimize=True)
                    
                    contador_global[0] += 1
                    procesadas += 1
                except Exception:
                    continue  # Imagen corrupta, saltar
    except Exception as e:
        print(f'  ⚠️ Error leyendo ZIP: {e}')
    
    return procesadas


def extraer_de_rar(rar_path, destino_extract, max_imgs, contador_global):
    """
    Extrae imágenes de un RAR usando unrar.
    Estrategia: extraer a carpeta temporal, procesar, limpiar.
    """
    procesadas = 0
    tmp_rar = os.path.join(destino_extract, 'rar_tmp')
    os.makedirs(tmp_rar, exist_ok=True)
    
    try:
        # Listar contenido del RAR primero
        result = subprocess.run(
            ['unrar', 'lb', rar_path],  # lb = listar solo nombres
            capture_output=True, text=True, timeout=60
        )
        
        if result.returncode != 0:
            print(f'  ⚠️ unrar no pudo listar {Path(rar_path).name}: {result.stderr[:100]}')
            return 0
        
        # Filtrar archivos de imagen en el RAR
        archivos_rar = [
            line.strip() for line in result.stdout.split('\n')
            if line.strip() and es_imagen(line.strip())
        ]
        random.shuffle(archivos_rar)
        
        # Calcular cuántas necesitamos de este RAR
        necesitamos = max_imgs - contador_global[0]
        archivos_a_extraer = archivos_rar[:necesitamos * 2]  # x2 por si hay corrompidas
        
        if not archivos_a_extraer:
            print(f'  ⚠️ No se encontraron imágenes en {Path(rar_path).name}')
            return 0
        
        print(f'  → Extrayendo {min(necesitamos, len(archivos_rar))} de {len(archivos_rar)} imgs...')
        
        # Extraer solo los archivos seleccionados
        for archivo in archivos_a_extraer:
            if contador_global[0] >= max_imgs:
                break
            
            try:
                subprocess.run(
                    ['unrar', 'e', '-o+', '-ep', rar_path, archivo, tmp_rar],
                    capture_output=True, timeout=30
                )
                
                # Buscar el archivo extraído
                nombre_base = Path(archivo).name
                ruta_extraida = os.path.join(tmp_rar, nombre_base)
                
                if os.path.exists(ruta_extraida):
                    idx = contador_global[0]
                    dest_path = os.path.join(DESTINO_CLASE, f'{CLASE}_{idx:06d}.jpg')
                    
                    if estandarizar_imagen(ruta_extraida, dest_path):
                        contador_global[0] += 1
                        procesadas += 1
                    
                    os.remove(ruta_extraida)  # Limpiar inmediatamente
            except Exception:
                continue
        
        # Limpiar carpeta temporal del RAR
        shutil.rmtree(tmp_rar, ignore_errors=True)
        
    except subprocess.TimeoutExpired:
        print(f'  ⚠️ Timeout procesando {Path(rar_path).name}')
    except Exception as e:
        print(f'  ⚠️ Error: {e}')
    
    return procesadas


print('✅ Funciones cargadas')

## CELDA 5 — Descarga y procesamiento de archivos

In [ ]:
# ── Pipeline principal ───────────────────────────────────────
# Requiere haber corrido CELDA 3B (PyDrive) para tener archivos_info

# Verificar cuántas imágenes ya tenemos (por si el notebook se interrumpió)
ya_tenemos = len([
    f for f in os.listdir(DESTINO_CLASE)
    if f.endswith('.jpg')
]) if os.path.exists(DESTINO_CLASE) else 0

print(f'Ya tenemos    : {ya_tenemos} imágenes en destino')
print(f'Meta          : {MAX_IMAGENES}')
print(f'Faltan        : {max(0, MAX_IMAGENES - ya_tenemos)}')

if ya_tenemos >= MAX_IMAGENES:
    print('✅ Ya tenemos suficientes imágenes. No se necesita procesar más.')
else:
    # Contador compartido (lista para pasarlo por referencia)
    contador = [ya_tenemos]
    
    print('\n' + '═' * 55)
    print('INICIANDO DESCARGA Y PROCESAMIENTO')
    print('═' * 55)
    
    # Ordenar: primero los más pequeños (más rápido de descargar)
    archivos_ordenados = sorted(archivos_info, key=lambda x: x['size_mb'])
    
    for archivo in archivos_ordenados:
        if contador[0] >= MAX_IMAGENES:
            print(f'\n✅ Meta alcanzada ({MAX_IMAGENES} imágenes). Deteniendo.')
            break
        
        print(f'\n📦 {archivo["nombre"]} ({archivo["size_mb"]:.1f} MB)')
        print(f'   Progreso: {contador[0]}/{MAX_IMAGENES} imágenes')
        
        # Descargar el archivo al directorio temporal
        ruta_local = os.path.join(TMP_DIR, archivo['nombre'])
        
        if not os.path.exists(ruta_local):
            print(f'   Descargando...')
            try:
                f_drive = drive_api.CreateFile({'id': archivo['id']})
                f_drive.GetContentFile(ruta_local)
                print(f'   ✅ Descargado')
            except Exception as e:
                print(f'   ❌ Error descargando: {e}')
                continue
        else:
            print(f'   (Ya estaba en caché local)')
        
        # Procesar según tipo
        ext = archivo['ext'].lower()
        antes = contador[0]
        
        if ext == '.zip':
            n = extraer_de_zip(ruta_local, TMP_EXTRACT, MAX_IMAGENES, contador)
        elif ext == '.rar':
            n = extraer_de_rar(ruta_local, TMP_EXTRACT, MAX_IMAGENES, contador)
        else:
            print(f'   ⚠️ Formato no soportado: {ext}')
            continue
        
        print(f'   ✅ +{n} imágenes procesadas (total: {contador[0]})')
        
        # Eliminar archivo descargado para liberar espacio en /content/
        os.remove(ruta_local)
        print(f'   🗑️  Archivo temporal eliminado')
    
    print(f'\n{"═" * 55}')
    print(f'RESUMEN FINAL')
    print(f'  Imágenes en destino : {contador[0]}')
    print(f'  Meta                : {MAX_IMAGENES}')
    print(f'  Estado              : {"✅ COMPLETO" if contador[0] >= MAX_IMAGENES else "⚠️ INCOMPLETO"}')
    print(f'  Destino             : {DESTINO_CLASE}')

## CELDA 6 — Verificación final

In [ ]:
# ── Verificar el resultado final ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imagenes = sorted([
    os.path.join(DESTINO_CLASE, f)
    for f in os.listdir(DESTINO_CLASE)
    if f.endswith('.jpg')
])

print(f'Total imágenes en {DESTINO_CLASE}: {len(imagenes)}')

if imagenes:
    # Verificar que no haya imágenes corruptas (sample de 50)
    sample = random.sample(imagenes, min(50, len(imagenes)))
    corruptas = 0
    for ruta in sample:
        try:
            img = Image.open(ruta)
            assert img.size == (224, 224)
            assert img.mode == 'RGB'
        except Exception:
            corruptas += 1
    
    print(f'Verificación (muestra 50): {50-corruptas}/50 válidas')
    
    if corruptas > 0:
        print(f'⚠️ {corruptas} imágenes con problemas detectadas')
    else:
        print(f'✅ Todas las imágenes son 224x224 RGB correctas')
    
    # Mostrar muestra visual (primeras 8 imágenes)
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    fig.suptitle(f'Muestra dataset: {CLASE} ({len(imagenes)} imgs)', fontsize=13)
    
    muestra_visual = random.sample(imagenes, min(8, len(imagenes)))
    for ax, ruta in zip(axes.flat, muestra_visual):
        try:
            img = mpimg.imread(ruta)
            ax.imshow(img)
            ax.set_title(Path(ruta).name[:15], fontsize=7)
        except Exception:
            ax.text(0.5, 0.5, 'Error', ha='center')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print(f'\n🎯 Listo para usar en el notebook de entrenamiento.')
print(f'   Ruta: {DESTINO_CLASE}')

## CELDA 7 — Limpieza (corre al terminar)

In [ ]:
# ── Limpiar archivos temporales ──────────────────────────────
# Corre esto cuando termines para liberar espacio en /content/

if os.path.exists(TMP_DIR):
    shutil.rmtree(TMP_DIR)
    print(f'✅ Limpiado: {TMP_DIR}')

print('\n📊 Espacio en disco actual:')
!df -h /content | tail -1